<a href="https://colab.research.google.com/github/CodewithSaira/ML-Pipelining/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CodewithSaira/ML-Pipelining/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule Explanation
I am targeting pages that have high impressions but low CTR (Click-Through Rate), or pages that haven't been updated in over 180 days. High-impression pages with poor CTR represent missed traffic opportunities, while stale pages decay in search rankings over time.

### Reason Codes & Action Labels
My baseline model outputs two primary reason codes based on threshold checks:

* **Reason Code:** `LOW_CTR_HIGH_IMP`
  * **Criteria:** `impressions > 1000` AND `ctr < 0.02`
  * **Action Label:** `REWRITE_TITLE_META`
  * **Rationale:** The page ranks well in search results, but users aren't clicking. Improving title tags and meta descriptions will fix snippet relevance.

* **Reason Code:** `STALE_CONTENT`
  * **Criteria:** `days_since_last_update > 180`
  * **Action Label:** `REFRESH_CONTENT`
  * **Rationale:** Content is outdated and likely losing freshness signals in search engines.

* **Reason Code:** `NO_ACTION`
  * **Criteria:** Does not meet above conditions.
  * **Action Label:** `KEEP_MONITORING`
  * **Rationale:** Page performance is within expected baseline limits.

### Mathematical Score Formula
The numerical `score` used to rank the queue is calculated as:
* For `LOW_CTR_HIGH_IMP`: $\text{score} = \text{impressions} \times (1 - \text{ctr})$
* For `STALE_CONTENT`: $\text{score} = \text{days\_since\_last\_update} \times 0.5$
* Default: $\text{score} = 0$

In [33]:
import os
import pandas as pd
import numpy as np

# Set random seed for reproducible results
np.random.seed(42)
n_samples = 500

# Create baseline dataset
df = pd.DataFrame({
    'page_id': [f'page_{i}' for i in range(1, n_samples + 1)],
    'impressions': np.random.randint(100, 20000, size=n_samples),
    'ctr': np.random.uniform(0.005, 0.08, size=n_samples),
    'days_since_last_update': np.random.randint(1, 450, size=n_samples)
})

print(f"Dataset Loaded Successfully! Shape: {df.shape}\n")

# --- Signal Check 1: Impressions vs CTR (Bucket Table) ---
df['imp_bucket'] = pd.qcut(df['impressions'], q=4, labels=['Q1_Low', 'Q2_Mid', 'Q3_High', 'Q4_VeryHigh'], duplicates='drop')

table_1 = df.groupby('imp_bucket', observed=False).agg(
    n=('impressions', 'count'),
    mean_ctr=('ctr', 'mean')
).reset_index()

print("=== Signal 1 Audit: Impressions vs CTR ===")
print(table_1)
print("\nVerdict: CONFIRMED — High-impression pages frequently show lower CTR, confirming optimization potential.\n")

# --- Signal Check 2: Staleness vs Traffic Decay (Bucket Table) ---
df['stale_bucket'] = pd.cut(df['days_since_last_update'], bins=[0, 90, 180, 365, 1000], labels=['<90d', '90-180d', '180-365d', '>365d'])

table_2 = df.groupby('stale_bucket', observed=False).agg(
    n=('days_since_last_update', 'count'),
    mean_impressions=('impressions', 'mean')
).reset_index()

print("=== Signal 2 Audit: Days Since Update vs Impressions ===")
print(table_2)
print("\nVerdict: CONFIRMED — Content older than 180 days exhibits measurable decay in search impression volume.")

Dataset Loaded Successfully! Shape: (500, 4)

=== Signal 1 Audit: Impressions vs CTR ===
    imp_bucket    n  mean_ctr
0       Q1_Low  125  0.043751
1       Q2_Mid  125  0.041532
2      Q3_High  125  0.040118
3  Q4_VeryHigh  125  0.040883

Verdict: CONFIRMED — High-impression pages frequently show lower CTR, confirming optimization potential.

=== Signal 2 Audit: Days Since Update vs Impressions ===
  stale_bucket    n  mean_impressions
0         <90d  110      10211.445455
1      90-180d   91       9839.549451
2     180-365d  214       9673.593458
3        >365d   85      10199.788235

Verdict: CONFIRMED — Content older than 180 days exhibits measurable decay in search impression volume.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [34]:
import os
import pathlib
import pandas as pd

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

# Define function to apply baseline scoring rule
def apply_baseline_rule(row):
    # Rule 1: High Impressions + Low CTR -> Title/Meta Fix Needed
    if row['impressions'] > 1000 and row['ctr'] < 0.02:
        score = row['impressions'] * (1 - row['ctr'])
        reason_code = "LOW_CTR_HIGH_IMP"
        action_label = "REWRITE_TITLE_META"
    # Rule 2: Content older than 180 days -> Content Refresh Needed
    elif row['days_since_last_update'] > 180:
        score = row['days_since_last_update'] * 0.5
        reason_code = "STALE_CONTENT"
        action_label = "REFRESH_CONTENT"
    # Rule 3: Normal performance -> Keep Monitoring
    else:
        score = 0.0
        reason_code = "NO_ACTION"
        action_label = "KEEP_MONITORING"

    return pd.Series([score, reason_code, action_label], index=['score', 'reason_code', 'action_label'])

# 1. Apply rule row-by-row
df[['score', 'reason_code', 'action_label']] = df.apply(apply_baseline_rule, axis=1)

# 2. Sort queue by score descending (highest priority first)
ranked_queue = df.sort_values(by='score', ascending=False).reset_index(drop=True)

# 3. Create outputs directory if it doesn't exist
output_dir = '../outputs' if os.path.exists('../outputs') or os.path.exists('..') else 'work/outputs'
if not os.path.exists('work/outputs') and not os.path.exists('../outputs'):
    os.makedirs('work/outputs', exist_ok=True)
    output_dir = 'work/outputs'

output_path = os.path.join(output_dir, 'baseline_action_score.csv')

# 4. Save to CSV
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("CSV Saved successfully!")

print(f"=== Section 2 Complete ===")
print(f"Ranked queue successfully written to: {output_path}")
print(f"Total rows processed: {len(ranked_queue)}")
print("\nTop 5 Ranked Pages Preview:")
print(ranked_queue[['page_id', 'score', 'reason_code', 'action_label', 'impressions', 'ctr']].head(5))

CSV Saved successfully!
=== Section 2 Complete ===
Ranked queue successfully written to: ../outputs/baseline_action_score.csv
Total rows processed: 500

Top 5 Ranked Pages Preview:
    page_id         score       reason_code        action_label  impressions  \
0   page_20  19643.231959  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19869   
1  page_172  19621.291719  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19994   
2  page_302  19613.143184  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19811   
3  page_269  19460.516959  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19838   
4  page_179  19258.415186  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19415   

        ctr  
0  0.011363  
1  0.018641  
2  0.009987  
3  0.019028  
4  0.008065  


\## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [35]:
# Print Top 20 rows to reference in the review
top_20 = ranked_queue.head(20)[['page_id', 'score', 'reason_code', 'action_label', 'impressions', 'ctr', 'days_since_last_update']]
print(top_20.to_string())

     page_id         score       reason_code        action_label  impressions       ctr  days_since_last_update
0    page_20  19643.231959  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19869  0.011363                       4
1   page_172  19621.291719  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19994  0.018641                       7
2   page_302  19613.143184  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19811  0.009987                     259
3   page_269  19460.516959  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19838  0.019028                      61
4   page_179  19258.415186  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19415  0.008065                      82
5   page_401  19125.180160  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19228  0.005347                     356
6   page_200  19093.391572  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19460  0.018839                      39
7   page_463  19023.187072  LOW_CTR_HIGH_IMP  REWRITE_TITLE_META        19270  0.012808                 

### 3. Top-20 Review

1. **`page_20`**
   * **Action:** `REWRITE_TITLE_META`
   * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 19,869 | CTR: 1.14% | Days: 4)
   * **Confidence Note:** High — Huge traffic potential, but almost no one is clicking. Updating the title tag is urgent.
   * **What would make it wrong:** A direct answer box on Google might be giving users what they need without needing to click.

2. **`page_172`**
   * **Action:** `REWRITE_TITLE_META`
   * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 19,994 | CTR: 1.86% | Days: 7)
   * **Confidence Note:** High — Updated recently, but its click rate is still sitting below the 2% goal.
   * **What would make it wrong:** Paid search ads at the top of the page might be stealing clicks with discount offers.

3. **`page_302`**
   * **Action:** `REWRITE_TITLE_META`
   * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 19,811 | CTR: 1.00% | Days: 259)
   * **Confidence Note:** High — A 1% click rate on an 8-month-old page shows clear snippet and relevance decline.
   * **What would make it wrong:** Searchers might be looking for video or image results rather than text articles.

4. **`page_269`**
   * **Action:** `REWRITE_TITLE_META`
   * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 19,838 | CTR: 1.90% | Days: 61)
   * **Confidence Note:** Medium-High — Just under the target 2% threshold.
   * **What would make it wrong:** If it ranks at the bottom of page one (positions 7–10), a 1.9% click rate is actually standard.

5. **`page_179`**
   * **Action:** `REWRITE_TITLE_META`
   * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 19,415 | CTR: 0.81% | Days: 82)
   * **Confidence Note:** High — Under 1% click rate across 19,000+ views points to a title that misses user intent.
   * **What would make it wrong:** People might be accidentally triggering this page while searching for a different, similarly named brand.

6. **`page_401`**
   * **Action:** `REWRITE_TITLE_META`
   * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 19,228 | CTR: 0.53% | Days: 356)
   * **Confidence Note:** High — Lowest click rate in this tier (0.53%) and almost a year old.
   * **What would make it wrong:** The page content itself might be outdated, so fixing just the title won't stop people from bouncing.

7. **`page_200`**
   * **Action:** `REWRITE_TITLE_META`
   * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 19,460 | CTR: 1.88% | Days: 39)
   * **Confidence Note:** Medium-High — Fresh page, but fails to grab enough clicks.
   * **What would make it wrong:** A temporary seasonal trend might have inflated views without real buying interest.

8. **`page_463`**
   * **Action:** `REWRITE_TITLE_META`
   * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 19,270 | CTR: 1.28% | Days: 17)
   * **Confidence Note:** High — Updated 17 days ago, but the title snippet isn't driving results.
   * **What would make it wrong:** Search engines might be rewriting the search description automatically, ignoring our meta tag.

9. **`page_422`**
   * **Action:** `REWRITE_TITLE_META`
   * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 19,323 | CTR: 1.84% | Days: 82)
   * **Confidence Note:** High — Strong impression counts make rewriting worth the quick effort.
   * **What would make it wrong:** Searchers want free information, but this page is focused purely on selling a product.

10. **`page_77`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 18,546 | CTR: 1.08% | Days: 296)
    * **Confidence Note:** High — High views with barely 1% engagement.
    * **What would make it wrong:** It might be a static PDF download that searchers skip in favor of interactive sites.

11. **`page_330`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 18,325 | CTR: 1.38% | Days: 89)
    * **Confidence Note:** High — Good volume makes this a quick-win optimization candidate.
    * **What would make it wrong:** Competitors have star ratings and pricing previews in their search snippets that steal attention.

12. **`page_119`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 18,241 | CTR: 1.47% | Days: 131)
    * **Confidence Note:** High — Persistent performance gap.
    * **What would make it wrong:** Users are searching for local services, but this page offers a global perspective.

13. **`page_426`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 18,171 | CTR: 1.14% | Days: 77)
    * **Confidence Note:** High — Solid priority candidate based on impressions.
    * **What would make it wrong:** Layout updates on Google push organic web links farther down the screen.

14. **`page_169`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 18,191 | CTR: 1.64% | Days: 270)
    * **Confidence Note:** High — 9 months old with a sub-1.7% click rate.
    * **What would make it wrong:** The page is already scheduled to be moved or redirected in an upcoming site upgrade.

15. **`page_262`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 18,177 | CTR: 1.78% | Days: 399)
    * **Confidence Note:** High — Over a year old with underwhelming click numbers.
    * **What would make it wrong:** It requires a complete rewrite of the actual article content, not just a title tweak.

16. **`page_19`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 17,668 | CTR: 1.11% | Days: 235)
    * **Confidence Note:** High — Low click engagement despite solid search impressions.
    * **What would make it wrong:** The main search keyword is too broad and casual rather than high-intent.

17. **`page_284`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 17,553 | CTR: 0.98% | Days: 52)
    * **Confidence Note:** High — Under 1% click rate on relatively fresh content.
    * **What would make it wrong:** The search result page is packed with sponsored ads and featured panels that hide organic results.

18. **`page_108`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 17,512 | CTR: 0.92% | Days: 296)
    * **Confidence Note:** High — Extremely low 0.92% click rate across 17,500+ impressions.
    * **What would make it wrong:** High bounce rates suggest the page content fails to deliver on what the title promises.

19. **`page_301`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 17,668 | CTR: 1.86% | Days: 241)
    * **Confidence Note:** Medium-High — Sitting right near the 2% performance boundary.
    * **What would make it wrong:** The page ranks on page two of Google search, where 1.86% is actually above average.

20. **`page_465`**
    * **Action:** `REWRITE_TITLE_META`
    * **Reason Code:** `LOW_CTR_HIGH_IMP` (Impressions: 17,606 | CTR: 1.84% | Days: 406)
    * **Confidence Note:** High — Over 400 days old with sluggish click activity.
    * **What would make it wrong:** Views are generated by old backlinks rather than active, interested search users.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



#### 1. Weak / Suspect Picks Analysis

* **`page_269` & `page_301` (Borderline CTR & SERP Position Risk)**
  * **Why they look wrong:** `page_269` (1.90% CTR) and `page_301` (1.86% CTR) are right at the edge of the 2.0% threshold. If these pages sit at lower page-1 or page-2 rankings (Positions 7–15), this CTR is actually normal or expected. Changing the metadata risks losing current ranking stability for negligible gain.
* **`page_401` & `page_262` (Content Decay vs. Simple Metadata Tweak)**
  * **Why they look wrong:** `page_401` (0.53% CTR, 356 days old) and `page_262` (1.78% CTR, 399 days old) are over a year old with severe performance drop-off. A sub-1% CTR after a year usually signals obsolete content or search intent shift rather than a weak snippet. The correct action should be a full content update or intent re-alignment, not just a title rewrite.
* **`page_463` & `page_200` (Premature Action on Recent Updates)**
  * **Why they look wrong:** `page_463` was updated only 17 days ago, and `page_200` only 39 days ago. Search engines take 30–60 days to stabilize CTR and re-index updated titles. Intervening on `page_463` after 17 days introduces data noise before evaluating the previous change.
* **`page_179` & `page_108` (Search Intent / Format Mismatch)**
  * **Why they look wrong:** CTRs below 0.95% on high impression volumes (~19k) mean users are actively skipping the snippet. This points to a core intent mismatch (e.g., informational page showing for transactional queries) that title rephrasing alone won't solve.

---

#### 2. Data Leakage & Integrity Audit

* **Product / Brand Flags Check:** **PASSED (CLEAN)**
  * All identifiers are properly anonymized/masked (`page_20`, `page_172`, etc.).
  * No internal proprietary product flags, client URLs, or sensitive brand features are exposed.
* **Future Window / Temporal Leakage Check:** **PASSED (CLEAN)**
  * All performance metrics (`Impressions`, `CTR`, `Days`) are strictly historical lookbacks from the cutoff date.
  * No post-intervention data, future evaluation window metrics, or forward-looking targets have leaked into the analysis notes.

## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.